In [1]:
import pandas as pd

arquivo_entrada = "06_PENSE2024_NUMERACAO_NOVA.parquet"

df = pd.read_parquet(arquivo_entrada)

print("Base carregada com sucesso!")
print(f"Linhas: {len(df):,}")
print(f"Colunas: {len(df.columns):,}")

Base carregada com sucesso!
Linhas: 148,725
Colunas: 41


In [2]:
# ============================================================
# REMOVER IDADES FORA DA FAIXA DE 13 A 17 ANOS
# ============================================================

coluna_idade = "IDADE_AGREG - IDADE DO ESTUDANTE, CATEGORIAS"


# Quantidade de linhas antes da remoção
linhas_antes = len(df)


# Manter somente códigos 2 e 3
df = df[
    df[coluna_idade]
    .astype(str)
    .str.strip()
    .isin(["2", "3"])
].copy()


# Quantidade de linhas depois da remoção
linhas_depois = len(df)


# Quantidade de linhas removidas
linhas_removidas = linhas_antes - linhas_depois


# Percentual de linhas removidas
percentual_removido = (
    linhas_removidas / linhas_antes
) * 100


# ============================================================
# MOSTRAR RESULTADOS
# ============================================================

print("========== FILTRO DE IDADE ==========")
print(f"Linhas antes da remoção:  {linhas_antes:,}")
print(f"Linhas removidas:         {linhas_removidas:,}")
print(f"Linhas restantes:         {linhas_depois:,}")
print(f"Percentual removido:      {percentual_removido:.2f}%")
print("======================================")


# ============================================================
# CONVERTER TODAS AS VARIÁVEIS PARA INTEIROS
# ============================================================
# Usa o tipo pandas "Int64" (I maiúsculo), que permite inteiros
# e valores ausentes (<NA>). Valores não numéricos ou decimais
# são identificados e geram erro para evitar perda silenciosa.

for coluna in df.columns:
    texto = df[coluna].astype("string").str.strip()

    # Tratar strings vazias como valores ausentes
    texto = texto.mask(texto.eq(""))

    numerico = pd.to_numeric(texto, errors="coerce")

    # Identificar valores preenchidos que não são numéricos
    invalidos = texto.notna() & numerico.isna()
    if invalidos.any():
        exemplos = texto[invalidos].drop_duplicates().head(5).tolist()
        raise ValueError(
            f"A coluna '{coluna}' contém valores não numéricos: {exemplos}"
        )

    # Verificar se os valores numéricos são inteiros
    nao_inteiros = numerico.notna() & ((numerico % 1) != 0)
    if nao_inteiros.any():
        exemplos = numerico[nao_inteiros].drop_duplicates().head(5).tolist()
        raise ValueError(
            f"A coluna '{coluna}' contém valores não inteiros: {exemplos}"
        )

    # Int64 permite valores inteiros e ausentes
    df[coluna] = numerico.astype("Int64")

print("\nTipos de dados após conversão:")
print(df.dtypes.value_counts().to_string())

# ============================================================
# GRAVAR NOVO ARQUIVO PARQUET
# ============================================================

arquivo_saida = "07_PENSE2024_IDADE_13_A_17.parquet"

df.to_parquet(
    arquivo_saida,
    index=False
)

print()
print(f"Arquivo salvo com sucesso: {arquivo_saida}")

========== FILTRO DE IDADE ==========
Linhas antes da remoção:  148,725
Linhas removidas:         30,684
Linhas restantes:         118,041
Percentual removido:      20.63%

Tipos de dados após conversão:
Int64    41

Arquivo salvo com sucesso: 07_PENSE2024_IDADE_13_A_17.parquet


In [3]:
# ============================================================
# GERAR TABELA FINAL DOS RESULTADOS
# ============================================================

import pandas as pd
import re

from IPython.display import HTML, display


# ============================================================
# 1. ARQUIVO COM AS REGRAS DE CODIFICAÇÃO
# ============================================================

arquivo_excel = "01_Colunas_recomendadas_pense_2024_final.xlsx"


# ============================================================
# 2. LER O ARQUIVO EXCEL
# ============================================================

regras = pd.read_excel(
    arquivo_excel,
    header=1
)


# ============================================================
# 3. NOMES DAS COLUNAS DO EXCEL
# ============================================================

coluna_codigo = "Código da coluna"
coluna_codificacao = "Nova codificacao"


if coluna_codigo not in regras.columns:
    raise ValueError(
        f"Coluna '{coluna_codigo}' não encontrada no Excel."
    )


if coluna_codificacao not in regras.columns:
    raise ValueError(
        f"Coluna '{coluna_codificacao}' não encontrada no Excel."
    )


# ============================================================
# 4. FUNÇÃO PARA NORMALIZAR OS CÓDIGOS
# ============================================================

def normalizar_codigo(valor):

    if pd.isna(valor):
        return None

    texto = str(valor).strip()

    if texto == "":
        return None

    try:

        numero = float(texto)

        if numero.is_integer():
            return str(int(numero))

    except (ValueError, TypeError):

        pass

    return texto


# ============================================================
# 5. FUNÇÃO PARA CRIAR O MAPA DA "NOVA CODIFICACAO"
# ============================================================

def criar_mapa_codificacao(codificacao):

    mapa = {}

    if pd.isna(codificacao):
        return mapa

    texto = str(codificacao).strip()

    if texto == "":
        return mapa


    # Pode haver vírgulas ou quebras de linha
    partes = re.split(
        r",|\n",
        texto
    )


    for parte in partes:

        parte = parte.strip()

        if not parte:
            continue

        if "=" not in parte:
            continue


        codigo, legenda = parte.split(
            "=",
            1
        )


        codigo = normalizar_codigo(codigo)

        legenda = legenda.strip()


        if codigo is not None and legenda:

            mapa[codigo] = legenda


    return mapa


# ============================================================
# 6. CRIAR MAPA:
#
# CÓDIGO DA COLUNA → NOVA CODIFICACAO
# ============================================================

mapas_codificacao = {}


for _, linha in regras.iterrows():

    codigo = linha[coluna_codigo]

    if pd.isna(codigo):
        continue

    codigo = str(codigo).strip()


    codificacao = linha[coluna_codificacao]

    if pd.isna(codificacao):
        continue


    mapa = criar_mapa_codificacao(
        codificacao
    )


    if mapa:

        mapas_codificacao[codigo] = mapa


# ============================================================
# 7. GERAR A TABELA RESUMO
# ============================================================

tabela_dados = []


for col in df.columns:

    col_limpa = str(col).strip()


    # ========================================================
    # IDENTIFICAR O CÓDIGO DA VARIÁVEL
    # ========================================================

    if " - " in col_limpa:

        codigo_variavel, descricao_variavel = (
            col_limpa.split(
                " - ",
                1
            )
        )

        codigo_variavel = codigo_variavel.strip()
        descricao_variavel = descricao_variavel.strip()

    else:

        codigo_variavel = col_limpa
        descricao_variavel = col_limpa


    # ========================================================
    # PROCURAR A CODIFICAÇÃO NO ARQUIVO 01
    # ========================================================

    mapa = mapas_codificacao.get(
        codigo_variavel,
        {}
    )


    # ========================================================
    # CONTAR AS RESPOSTAS
    # ========================================================

    counts = df[col].value_counts(
        dropna=False
    )


    total = len(df)

    respostas_formatadas = []


    # ========================================================
    # FORMATAR CADA RESPOSTA
    # ========================================================

    for valor, quantidade in counts.items():

        pct = (
            quantidade / total
        ) * 100


        # ----------------------------------------------------
        # VALORES NULOS / EM BRANCO
        # ----------------------------------------------------

        if pd.isna(valor):

            respostas_formatadas.append(
                f"[NaN] Valores Nulos / Em Branco (NaN): "
                f"{pct:.2f}%"
            )

            continue


        # ----------------------------------------------------
        # NORMALIZAR O CÓDIGO DA RESPOSTA
        # ----------------------------------------------------

        codigo_resposta = normalizar_codigo(
            valor
        )


        # ----------------------------------------------------
        # PROCURAR A LEGENDA
        # ----------------------------------------------------

        legenda = mapa.get(
            codigo_resposta
        )


        # ----------------------------------------------------
        # MONTAR O RESULTADO
        # ----------------------------------------------------

        if legenda is not None:

            respostas_formatadas.append(
                f"[{codigo_resposta}] "
                f"{pct:.2f}% - {legenda}"
            )

        else:

            respostas_formatadas.append(
                f"[{codigo_resposta}] "
                f"{pct:.2f}%"
            )


    # ========================================================
    # ADICIONAR À TABELA
    # ========================================================

    tabela_dados.append({

        "Pergunta / Variável":
            col_limpa,

        "Distribuição das Respostas":
            "\n".join(
                respostas_formatadas
            )
    })


# ============================================================
# 8. CRIAR DATAFRAME DA TABELA
# ============================================================

df_tabela = pd.DataFrame(
    tabela_dados
)


# ============================================================
# 9. ESTILIZAR A TABELA
# ============================================================

styler = df_tabela.style.set_properties(
    **{
        "white-space": "pre-wrap",
        "text-align": "left"
    }
)


# ============================================================
# 10. EXIBIR
# ============================================================

print(
    "========== TABELA FINAL DOS RESULTADOS ==========\n"
)

display(
    HTML(
        styler.to_html()
    )
)

========== TABELA FINAL DOS RESULTADOS ==========



,Pergunta / Variável,Distribuição das Respostas
0,REGIAO - código de regiões,[2] 33.23% - nordeste [1] 24.29% - norte [3] 18.60% - sudeste [5] 12.62% - centro-oeste [4] 11.26% - sul
1,UF - código UF,[32] 5.06% - Espírito Santo [35] 4.82% - São Paulo [29] 4.58% - Bahia [26] 4.57% - Pernambuco [33] 4.54% - Rio de Janeiro [31] 4.18% - Minas Gerais [42] 4.11% - Santa Catarina [11] 4.07% - Rondônia [17] 4.03% - Tocantins [41] 3.97% - Paraná [28] 3.86% - Sergipe [50] 3.81% - Mato Grosso do Sul [21] 3.79% - Maranhão [27] 3.74% - Alagoas [13] 3.62% - Amazonas [25] 3.57% - Paraíba [15] 3.49% - Pará [12] 3.34% - Acre [22] 3.24% - Piauí [52] 3.20% - Goiás [43] 3.18% - Rio Grande do Sul [51] 3.16% - Mato Grosso [16] 3.12% - Amapá [24] 2.95% - Rio Grande do Norte [23] 2.94% - Ceará [14] 2.63% - Roraima [53] 2.46% - Distrito Federal
2,CAPITAL - indica se é município da capital,[1] 51.79% - Capital [0] 48.21%
3,B01001A - Qual é o seu sexo?,[2] 50.15% - Mulher [1] 49.85% - Homem
4,"IDADE_AGREG - IDADE DO ESTUDANTE, CATEGORIAS",[2] 65.03% - 13 a 15 anos [3] 34.97% - 16 e 17 anos
5,B01002 - Qual é a sua cor ou raça?,[4] 42.90% - Parda [1] 41.01% - Branca [2] 10.79% - Preta [3] 2.71% - Amarela [5] 2.13% - Indígena [0] 0.45% - Sem reposta
6,B01006 - Você mora com sua mãe?,[3] 89.80% - Sim [1] 10.16% - Nao [0] 0.04% - Sem resposta
7,B01007 - Você mora com seu pai?,[3] 62.99% - Sim [1] 36.93% - Nao [0] 0.08% - Sem resposta
8,B01015B - Na sua casa tem computador ou notebook?,[3] 63.57% - Sim [1] 36.36% - Nao [0] 0.07% - Sem resposta
9,B01016 - Você tem acesso à internet em sua casa?,[3] 97.40% - Sim [1] 2.56% - Nao [0] 0.04% - Sem resposta


In [4]:
# ============================================================
# CONTAR LINHAS DE ACORDO COM A QUANTIDADE DE VALORES "0"
# ============================================================

# Transformar os valores em string para garantir que
# 0, 0.0 e "0" sejam tratados da mesma forma.

df_string = df.astype(str)


# Identificar onde existe o valor "0"
mascara_zero = (
    df_string
    .apply(
        lambda coluna:
        coluna.str.strip().isin(["0", "0.0"])
    )
)


# Contar quantos valores "0" existem em cada linha
quantidade_zeros_por_linha = mascara_zero.sum(axis=1)


# ============================================================
# CONTAGEM
# ============================================================

total_linhas = len(df)

linhas_maior_5 = (
    quantidade_zeros_por_linha > 5
).sum()

linhas_4 = (
    quantidade_zeros_por_linha == 4
).sum()

linhas_3 = (
    quantidade_zeros_por_linha == 3
).sum()

linhas_2 = (
    quantidade_zeros_por_linha == 2
).sum()

linhas_1 = (
    quantidade_zeros_por_linha == 1
).sum()

linhas_0 = (
    quantidade_zeros_por_linha == 0
).sum()


# ============================================================
# MOSTRAR RESULTADOS
# ============================================================

print("========== QUANTIDADE DE VALORES '0' POR LINHA ==========")

print(
    f"Linhas com >5 valores '0': "
    f"{linhas_maior_5:,} "
    f"({linhas_maior_5 / total_linhas * 100:.2f}%)"
)

print(
    f"Linhas com 4 valores '0':  "
    f"{linhas_4:,} "
    f"({linhas_4 / total_linhas * 100:.2f}%)"
)

print(
    f"Linhas com 3 valores '0':  "
    f"{linhas_3:,} "
    f"({linhas_3 / total_linhas * 100:.2f}%)"
)

print(
    f"Linhas com 2 valores '0':  "
    f"{linhas_2:,} "
    f"({linhas_2 / total_linhas * 100:.2f}%)"
)

print(
    f"Linhas com 1 valor '0':    "
    f"{linhas_1:,} "
    f"({linhas_1 / total_linhas * 100:.2f}%)"
)

print(
    f"Linhas com 0 valores '0':  "
    f"{linhas_0:,} "
    f"({linhas_0 / total_linhas * 100:.2f}%)"
)

print("==========================================================")
print(f"Total de linhas: {total_linhas:,}")

========== QUANTIDADE DE VALORES '0' POR LINHA ==========
Linhas com >5 valores '0': 1,283 (1.09%)
Linhas com 4 valores '0':  1,742 (1.48%)
Linhas com 3 valores '0':  10,715 (9.08%)
Linhas com 2 valores '0':  15,230 (12.90%)
Linhas com 1 valor '0':    44,687 (37.86%)
Linhas com 0 valores '0':  43,890 (37.18%)
Total de linhas: 118,041


In [5]:
# Verificar tipos de dados do arquivo de saida parquet
import pandas as pd
from IPython.display import display, HTML

# ============================================================
# 1. ARQUIVO DE SAÍDA
# ============================================================

arquivo_saida = "07_PENSE2024_IDADE_13_A_17.parquet"

# ============================================================
# 2. LER O ARQUIVO
# ============================================================

df_saida = pd.read_parquet(arquivo_saida)

print(f"Arquivo: {arquivo_saida}")
print(f"Linhas: {len(df_saida):,}")
print(f"Colunas: {len(df_saida.columns)}")

# ============================================================
# 3. CRIAR RELATÓRIO DOS TIPOS DE DADOS
# ============================================================

relatorio_tipos = pd.DataFrame({
    "Coluna": df_saida.columns,
    "Tipo Pandas": df_saida.dtypes.astype(str).values,
    "Valores não nulos": df_saida.count().values,
    "Valores nulos": df_saida.isna().sum().values,
    "Valores distintos": df_saida.nunique(dropna=True).values
})

# ============================================================
# 4. CONVERTER A TABELA PARA HTML
# ============================================================

html = relatorio_tipos.to_html(
    index=False,
    escape=True,
    border=0,
    classes="tipos-table"
)

# ============================================================
# 5. EXIBIR TABELA COM ROLAGEM E TONS DE CINZA
# ============================================================

display(HTML(f"""
<div style="
    width: 100%;
    max-width: 100%;
    max-height: 420px;
    overflow: auto;
    border: 1px solid #bdbdbd;
    border-radius: 6px;
    background-color: #e8e8e8;
">

<style>
.tipos-table {{
    border-collapse: collapse;
    width: 100%;
    font-size: 12px;
    color: #333333;
    background-color: #e8e8e8;
}}

.tipos-table th {{
    position: sticky;
    top: 0;
    z-index: 2;
    background-color: #d0d0d0;
    color: #333333;
    padding: 8px;
    text-align: left;
    white-space: nowrap;
    border-bottom: 2px solid #b0b0b0;
}}

.tipos-table td {{
    padding: 7px;
    border-bottom: 1px solid #c8c8c8;
    overflow-wrap: anywhere;
    background-color: #e8e8e8;
}}

.tipos-table tr:nth-child(even) td {{
    background-color: #dedede;
}}

.tipos-table tr:hover td {{
    background-color: #d3d3d3;
}}
</style>

{html}

</div>
"""))


Arquivo: 07_PENSE2024_IDADE_13_A_17.parquet
Linhas: 118,041
Colunas: 41


Coluna,Tipo Pandas,Valores não nulos,Valores nulos,Valores distintos
REGIAO - código de regiões,Int64,118041,0,5
UF - código UF,Int64,118041,0,27
CAPITAL - indica se é município da capital,Int64,118041,0,2
B01001A - Qual é o seu sexo?,Int64,118041,0,2
"IDADE_AGREG - IDADE DO ESTUDANTE, CATEGORIAS",Int64,118041,0,2
B01002 - Qual é a sua cor ou raça?,Int64,118041,0,6
B01006 - Você mora com sua mãe?,Int64,118041,0,3
B01007 - Você mora com seu pai?,Int64,118041,0,3
B01015B - Na sua casa tem computador ou notebook?,Int64,118041,0,3
B01016 - Você tem acesso à internet em sua casa?,Int64,118041,0,3


In [6]:

import pandas as pd
from IPython.display import display, HTML

# Arquivo Parquet gerado pelo script 06
arquivo = "07_PENSE2024_IDADE_13_A_17.parquet"

# Ler o arquivo Parquet
df_saida = pd.read_parquet(arquivo)

# Informações gerais
print(f"Arquivo: {arquivo}")
print(f"Linhas: {len(df_saida):,}")
print(f"Colunas: {len(df_saida.columns)}")

# Relatório dos tipos de dados
relatorio_tipos = pd.DataFrame({
    "Variável": df_saida.columns,
    "Tipo Pandas": df_saida.dtypes.astype(str).values,
    "Não nulos": df_saida.count().values,
    "Nulos": df_saida.isna().sum().values,
    "Valores distintos": df_saida.nunique(dropna=True).values
})

# Converter relatório para HTML
html = relatorio_tipos.to_html(
    index=False,
    escape=True,
    border=0,
    classes="tipos-table"
)

# Exibir tabela com rolagem e cabeçalho fixo
display(HTML(f"""
<div style="
    max-height: 420px;
    overflow: auto;
    border: 1px solid #bdbdbd;
    border-radius: 6px;
">
<style>
.tipos-table {{
    border-collapse: collapse;
    width: 100%;
    font-size: 12px;
    color: #333;
    background: #e8e8e8;
}}
.tipos-table th {{
    position: sticky;
    top: 0;
    z-index: 2;
    background: #d0d0d0;
    padding: 8px;
    text-align: left;
    white-space: nowrap;
    border-bottom: 2px solid #b0b0b0;
}}
.tipos-table td {{
    padding: 7px;
    border-bottom: 1px solid #c8c8c8;
    background: #e8e8e8;
    overflow-wrap: anywhere;
}}
.tipos-table tr:nth-child(even) td {{
    background: #dedede;
}}
.tipos-table tr:hover td {{
    background: #d3d3d3;
}}
</style>
{html}
</div>
"""))


Arquivo: 07_PENSE2024_IDADE_13_A_17.parquet
Linhas: 118,041
Colunas: 41


Variável,Tipo Pandas,Não nulos,Nulos,Valores distintos
REGIAO - código de regiões,Int64,118041,0,5
UF - código UF,Int64,118041,0,27
CAPITAL - indica se é município da capital,Int64,118041,0,2
B01001A - Qual é o seu sexo?,Int64,118041,0,2
"IDADE_AGREG - IDADE DO ESTUDANTE, CATEGORIAS",Int64,118041,0,2
B01002 - Qual é a sua cor ou raça?,Int64,118041,0,6
B01006 - Você mora com sua mãe?,Int64,118041,0,3
B01007 - Você mora com seu pai?,Int64,118041,0,3
B01015B - Na sua casa tem computador ou notebook?,Int64,118041,0,3
B01016 - Você tem acesso à internet em sua casa?,Int64,118041,0,3
